In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import gc
import yaml
import logging
from collections import defaultdict, deque
from typing import Any

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True 
)


In [2]:
class ConnectionManager():
    def __init__(self, db_con_str):
        # la fonction duckdb.connect transforme le path en absolue, autant le faire ici 
        self.db_con_str = Path(db_con_str).expanduser().resolve()

        # connexion lazy
        self._con = None


    @property
    def con(self):
        if(self._con is None):
            # se connecter à la base de données
            self._con = duckdb.connect(self.db_con_str)

        return self._con


    @con.setter
    def con(self, value):
        # si au moment de changer la connexion on a déjà une connexion active
        if(self._con is not None):
            self._con.close() # cloturer la connexion en cours
            self._con = None # retirer la référence sur la connexion en cours
            gc.collect() # appeler le garbage collector pour forcer l'action de libérer les ressources et éviter les conflits d'accès

        self._con = value # pointer sur la nouvelle connexion
    
    
    def close_con(self):
        # on exploite le setter de la propriété pour cloturer correctement la connexion
        self.con = None


    def __del__(self):
        """Ferme automatiquement la connexion DuckDB quand l'objet est détruit."""
        try:
            self.close_con()
        except Exception:
            pass


    def __enter__(self):
        return self
    

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close_con()
        return False

In [3]:
class ConnectionUtils(ConnectionManager):
    def __init__(self, db_con_str : str):
        super().__init__(db_con_str)


    def tables(self):
        """Retourne la liste de toutes les tables physiques de la base courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)


    def views(self):
        """Cette fonction renvoi la liste de toutes vues accessibles dans la base de données courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.views
            WHERE table_catalog = current_database()
            ORDER BY table_name
        """)


    def tables_views(self):
        """Retourne la liste des tables et des vues de la base courante"""
        return self.con.sql("""
            SELECT 
                table_name,
                table_type          -- 'BASE TABLE' ou 'VIEW'
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            ORDER BY table_type, table_name
        """)


    def table_exists(self, table_name : str):
        """Cette foction vérife qu'une table physique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{table_name}')
                AND
                (table_type = 'BASE TABLE')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]
    
    
    def view_exists(self, view_name : str):
        """Cette fonction check si une vue existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE 
                (table_name = '{view_name}')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def table_view_exists(self, name : str):
        """Cette foction vérife qu'une table physique ou une vue logique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{name}')
                AND
                (table_type = 'BASE TABLE' OR table_type = 'VIEW')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def drop_table_if_exists(self, table_name : str):
        """Cette fonction permet de supprimer une table s'elle existe"""
        self.con.sql(f"DROP TABLE IF EXISTS {table_name}")


    def drop_tables_if_exists(self, tables : list[str]):
        """Cette fonction surpprime chaque table de la liste tables s'elle existe dans la base courante"""
        for table_name in tables : 
            self.drop_table_if_exists(table_name)


    def drop_view_if_exists(self, view_name : str):
        """Cette fonction permet de supprimer une vue s'elle existe"""
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def drop_views_if_exists(self, views : list[str]):
        """Cette fonction surpprime chaque vue de la liste views s'elle existe dans la base courante"""
        for view_name in views : 
            self.drop_view_if_exists(view_name)


    def table(self, table_name : str):
        """Cette fonction renvoi la table dont le nom est passé en paramètre"""
        return self.con.table(table_name)


    def view(self, view_name : str):
        """Cette fonction renvoi la vue dont le nom est passé en paramètre"""
        return self.con.view(view_name)


    def table_view(self, name : str):
        """Retourne la relation d'une table ou d'une vue selon ce qui existe."""
        return self.con.sql(f"SELECT * FROM {name}")


    def create_table_view_if_not_exists(self, name: str, sql: str, type: str = "VIEW"):
        """
        Crée une vue ou une table uniquement si elle n'existe pas encore.
        """
        sql = sql.strip().rstrip(";")
        self.con.sql(f"""CREATE {type} IF NOT EXISTS {name} AS ({sql})""")
        

In [4]:
class DependencyTree:
    def __init__(self, data: dict[str, dict[str, Any]]):
        """
        data : dictionnaire de la forme
        {
            "v_sales": {"requires": ["t_sales"], ...},
            "t_sales": {"requires": ["df_sales"], ...},
            ...
        }
        """
        self.data = data
        self.graph = self._build_graph()          # node -> list of dependencies
        self.reverse_graph = self._build_reverse_graph()  # node -> list of dependents

    def _build_graph(self) -> dict[str, list[str]]:
        return {
            name: config.get("requires", [])
            for name, config in self.data.items()
        }

    def _build_reverse_graph(self) -> dict[str, list[str]]:
        reverse = defaultdict(list)
        for node, deps in self.graph.items():
            for dep in deps:
                reverse[dep].append(node)
        return dict(reverse)

    # -------------------------------------------------------------------------
    # Informations de base
    # -------------------------------------------------------------------------
    def nodes(self) -> list[str]:
        """Retourne tous les nœuds du graphe."""
        return list(self.graph.keys())

    def dependencies(self, name: str) -> list[str]:
        """Retourne les dépendances directes d'un nœud."""
        return self.graph.get(name, [])

    def dependents(self, name: str) -> list[str]:
        """Retourne les nœuds qui dépendent directement de celui-ci."""
        return self.reverse_graph.get(name, [])

    def roots(self) -> list[str]:
        """Nœuds qui ne sont requis par personne."""
        all_deps = {dep for deps in self.graph.values() for dep in deps}
        return [n for n in self.graph if n not in all_deps]

    def leaves(self) -> list[str]:
        """Nœuds qui n'ont aucune dépendance."""
        return [n for n, deps in self.graph.items() if not deps]

    # -------------------------------------------------------------------------
    # Dépendances récursives
    # -------------------------------------------------------------------------
    def all_dependencies(self, name: str) -> list[str]:
        """Retourne toutes les dépendances (directes + indirectes) dans l'ordre topologique."""
        result = []
        visited = set()

        def dfs(node: str):
            if node in visited:
                return
            visited.add(node)
            for dep in self.graph.get(node, []):
                dfs(dep)
            result.append(node)

        dfs(name)
        return result[:-1]  # on retire le nœud lui-même

    def creation_order(self, name: str | None = None) -> list[str]:
        """
        Ordre de création (topologique).
        Si name est fourni → uniquement pour ce nœud et ses dépendances.
        Sinon → ordre global.
        """
        if name:
            nodes = self.all_dependencies(name) + [name]
        else:
            nodes = self.nodes()

        in_degree = {n: 0 for n in nodes}
        for n in nodes:
            for dep in self.graph.get(n, []):
                if dep in in_degree:
                    in_degree[n] += 1

        queue = deque([n for n, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent in self.reverse_graph.get(node, []):
                if dependent in in_degree:
                    in_degree[dependent] -= 1
                    if in_degree[dependent] == 0:
                        queue.append(dependent)

        return order

    # -------------------------------------------------------------------------
    # Affichage
    # -------------------------------------------------------------------------
    def print_tree(self, root: str | None = None):
        """Affiche l'arbre de dépendances en texte."""
        def _print(node: str, prefix: str = "", is_last: bool = True, visited: set | None = None):
            if visited is None:
                visited = set()

            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")

            if node in visited:
                print(f"{prefix}{'    ' if is_last else '│   '}└── [cycle détecté]")
                return

            visited = visited | {node}
            deps = self.graph.get(node, [])
            new_prefix = prefix + ("    " if is_last else "│   ")

            for i, dep in enumerate(deps):
                _print(dep, new_prefix, i == len(deps) - 1, visited)

        if root:
            print(f"\n=== Dépendances de '{root}' ===\n")
            _print(root)
        else:
            print("\n=== Graphe complet ===\n")
            roots = self.roots()
            for i, r in enumerate(roots):
                _print(r, is_last=(i == len(roots) - 1))

    def print_levels(self, name: str | None = None):
        """Affiche les nœuds par niveau topologique."""
        order = self.creation_order(name)
        print(f"\n=== Ordre de création {'de ' + name if name else 'global'} ===\n")
        for i, node in enumerate(order, 1):
            print(f"{i:2d}. {node}")

In [5]:
class ConnectionPipeline(ConnectionUtils):
    def __init__(self, db_con_str : str, pipeline_file_path : str):
        super().__init__(db_con_str)
        self.pipeline_file_path = Path(pipeline_file_path).expanduser().resolve()
        self._pipeline = None
        self._tree = None


    def load_pipeline(self) -> dict:
        """Charge le fichier de définition des tables/vues."""
        with open(self.pipeline_file_path, "r", encoding="utf-8") as f:
            pipeline = yaml.safe_load(f)
            return pipeline


    @property
    def pipeline(self):
        if(self._pipeline is None):
            self._pipeline = self.load_pipeline()
        return self._pipeline


    @property
    def tree(self):
        if(self._tree is None):
            self._tree = DependencyTree(self.pipeline)
        return self._tree
    

    def df_from_file(self, file: str | Path, **kwargs) -> pd.DataFrame:
        """Charge un fichier en DataFrame selon son extension + options"""
        path = Path(file).expanduser().resolve()
        suffix = path.suffix.lower()

        if suffix in {".xlsx", ".xls", ".xlsm"}:
            return pd.read_excel(path, **kwargs)

        elif suffix == ".csv":
            return pd.read_csv(path, **kwargs)

        elif suffix == ".tsv":
            return pd.read_csv(path, sep="\t", **kwargs)

        elif suffix == ".json":
            return pd.read_json(path, **kwargs)

        elif suffix == ".parquet":
            return pd.read_parquet(path, **kwargs)

        else:
            raise ValueError(f"Extension non supportée : {suffix}")


    def df_from_file_config(self, config : dict):
        # On prépare les kwargs en enlevant les clés réservées
        reserved = {"type", "requires", "file"}
        kwargs = {k: v for k, v in config.items() if k not in reserved}

        return self.df_from_file(config["file"], **kwargs)


    def process_dataframe_type(self, name : str):
        if(name in self.pipeline):
            if(not self.table_view_exists(name)):
                config = self.pipeline[name]
                df = self.df_from_file_config(config)
                self.con.register(name, df)


    def process_table_view_type(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]
            self.create_table_view_if_not_exists(name, config["sql"], config["type"])


    def process(self, name : str):
        logging.getLogger().debug(f"process({name})")

        if(name in self.pipeline):
            config = self.pipeline[name]

            if(config["type"] == "dataframe"):
                self.process_dataframe_type(name)

            elif(config["type"] in ["table", "view"]):
                self.process_table_view_type(name)


    def process_with_requires(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]

            if(not self.table_view_exists(name)):
                for subname in config.get("requires", []):
                    self.process_with_requires(subname)

                self.process(name)


    def p_table_view(self, name : str):
        self.process_with_requires(name)
        return self.table_view(name)

In [6]:
class SalesPilBase(ConnectionPipeline):
    def __init__(self, db_con_str = "duckdb/pilotes/base/base.duckdb", pipeline_file_path = "config/base_pipeline.yaml"):
        super().__init__(db_con_str, pipeline_file_path)


    @classmethod
    def main(cls):
        """Cette fonction représente la fonction principale qui exploite cette classe et qu'elle faut exécuter"""

        base = SalesPilBase() # on instancie un objet de la classe qui gère les données pilote de base

        print(base.p_table_view("v_sales").df().shape) # on inspecte le shape de la dataframe des ventes brutes
        display(base.p_table_view("v_sales").df().head(3)) # on inspecte un echantillons de données de cette dataframe

        print(base.p_table_view("v_sales_model").df().shape) # on inspecte le shape de la dataframe des ventes brutes des donnéees de modélisation
        display(base.p_table_view("v_sales_model").df().head(3)) # on inspecte un echantillons de données de cette dataframe
        

        print(base.p_table_view("v_sales_model_base").df().shape) # on inspecte le shape de la dataframe de modélisaiton de base 
        display(base.p_table_view("v_sales_model_base").df().head(3)) # on inspecte un echantillons de données de cette dataframe

        base.close_con() # on cloture correctement la connexion pour éviter tout conflit


In [7]:
SalesPilBase.main() # exécuter la fonction principale de la classe permettant de construire la base de données de modélisation

(130566, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(109342, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(872, 103)


,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,nombre_mois,produit_nombre_ventes_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,NON-F&B,SOUVENIRS,Tote-Bag Parisienne,6.0,192.0,192.0,0.0,32.0,32.00,0.00,24,0.250000,8.00000,8.00000,0.000,13.0,3,329.0,329.0,-2.131628e-14,25.307692,25.307692,-1.639714e-15,4.333333,109.666667,109.666667,-7.105427e-15,0.541667,13.708333,13.708333,-8.881784e-16,218.0,48,5,3435.4402,3495.10225,-59.66205,15.758900,16.032579,-0.273679,4.541667,71.571671,72.814630,-1.242959,43.60,687.08804,699.02045,-11.93241,9.083333,143.143342,145.629260,-2.485919,15759.0,146,9,2,84373.3057,83953.10225,420.20345,5.353976,5.327312,0.026664,107.938356,577.899354,575.021248,2.878106,1751.000000,9374.811744,9328.122472,46.689272,7879.5,42186.65285,41976.551125,210.101725,0.041096,24.333333,2626.500000,14062.217617,13992.183708,70.033908,0.006849,656.625000,3515.554404,3498.045927,17.508477,0.020548,0.328767,0.000381,0.002276,0.002287,0.000000,0.000825,0.003899,0.003919,-5.072848e-17,0.013833,0.040717,0.041632,-0.141984
1,H2075,Ibis budget Nice,simply,6.0,NON-F&B,SOS,Spray Répulsif Anti Moustique et Tique Icaridine - 60 Ml,5.0,40.0,40.0,0.0,8.0,8.00,0.00,29,0.172414,1.37931,1.37931,0.000,25.0,5,251.5,281.0,-2.950000e+01,10.060000,11.240000,-1.180000e+00,5.000000,50.300000,56.200000,-5.900000e+00,0.862069,8.672414,9.689655,-1.017241e+00,730.0,97,5,7319.5600,7360.68000,-41.12000,10.026795,10.083123,-0.056329,7.525773,75.459381,75.883299,-0.423918,146.00,1463.91200,1472.13600,-8.22400,25.172414,252.398621,253.816552

In [10]:
class SalesPilSim(SalesPilBase):
    """
    Cette classe permet d'exécuter des simulations en relançant les mêmes calculs de la classe mère sur des vues de données modifiées en fonction de ce qu'on souhaite simuler.
    L'objectif de cette classe est de construire plusieurs jeux de données de simulation qui vont être exploitées par la suite par une modèle de machine learning.
    """
    def __init__(self, db_con_str = "duckdb/pilotes/sim/sim.duckdb", pipeline_file_path = "config/sim_pipeline.yaml"):
        super().__init__(db_con_str, pipeline_file_path)


    def remove_elements(self, champs : str, elements : list[str]):
        """Cette fonction permet de calculer les indicateurs de simulation si certains éléments (produits, gammes, types, etc) des données de ventes brutes n'existaient pas"""

        # convertir la liste des éléments à retirer en une chaine de caractères pour l'inclure dans la requête sql
        list_str = ",".join([f"'{elm}'" for elm in elements])

        # s'assurer que la table des données de modélisation existe déjà
        self.p_table_view("t_sales_model")

        # s'assurer que la table figée des indicateurs avant toute simulation existe déjà
        self.p_table_view("t_sales_model_base")

        # supprimer les vues afin de les recréer en partant de la vue source après avoir retiré les éléments
        self.drop_views_if_exists(self.views().df()["table_name"])

        # calculer la nouvelle quantité de demandes disponible correspondant aux clients acheteurs qui n'achetéront plus les produits retirés 
        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    COALESCE(SUM(CASE
                        WHEN {champs} IN ({list_str}) THEN QUANTITE
                        ELSE 0
                    END), 0) AS demande_quantite
                FROM
                    t_sales_model
                GROUP BY
                    hotel_code
                ORDER BY
                    hotel_code
            )
        """)

        # récupérer la part dans le nombre de ventes pour chaque produit vendu par chaque hotel et l'associer à la nouvelle quantité de demande
        self.con.sql("""
            CREATE OR REPLACE VIEW v_part_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    produit AS NOM_PRODUIT,
                    produit_nombre_ventes,
                    produit_part_nombre_ventes,
                    demande_quantite,
                    demande_quantite * produit_part_nombre_ventes AS part_demande_quantite
                FROM
                    v_demande_quantite
                LEFT JOIN
                    t_sales_model_base
                USING
                    (HOTEL_CODE)
                ORDER BY
                    HOTEL_CODE,
                    NOM_PRODUIT
            )
        """)

        # créer une liste de ventes fictives correspondant à la demande à intégrer et respectant le comportement des clients modélisés
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_demande AS (
                SELECT 
                    SOLUTION,
                    HOTEL_CODE,
                    HOTEL_NAME,
                    METRES_LINEAIRES,
                    NOM_BOUTIQUE, 
                    TYPE,
                    TYPE_RAW,
                    GAMME,
                    GAMME_RAW,
                    NOM_PRODUIT,
                    NOM_PRODUIT_RAW,
                    CATEGORIE,
                    OPERATEUR,
                    MACHINE,
                    DATE,
                    HEURE,
                    STATUT,
                    CODE_EAN,
                    (QUANTITE / produit_nombre_ventes) * part_demande_quantite AS QUANTITE,
                    (PRIX_HT / produit_nombre_ventes)  * part_demande_quantite AS PRIX_HT,
                    VAT,
                    (PRIX_TTC / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC,
                    MARQUE,
                    FOURNISSEUR,
                    - ORDER_ID AS ORDER_ID,
                    TEMPERATURE,
                    (PRIX_TTC_MARCHE / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC_MARCHE,
                    (MARGE / produit_nombre_ventes)  * part_demande_quantite AS MARGE
                FROM 
                    t_sales_model
                RIGHT JOIN
                (
                    SELECT
                        *
                    FROM
                        v_part_demande_quantite
                    WHERE
                        part_demande_quantite > 0
                )
                USING
                    (HOTEL_CODE, NOM_PRODUIT)
            )
            """)

        # créer la vue des données qui sera utilisée pour faire le calcul des indicateurs sur les données de modélisation 
        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_sales_model AS (
                SELECT
                    *
                FROM
                    t_sales_model
                WHERE
                    {champs} NOT IN ({list_str})

                UNION ALL

                SELECT
                    *
                FROM
                    v_sales_model_demande
                WHERE
                    {champs} NOT IN ({list_str})
            )
        """)

        # recréer la vue des indicateurs et toutes les vues intermédiaires en partant de la vue source modifiée
        self.p_table_view("v_sales_model_base")

        

        


    def remove_produits(self, produits : list[str]):
        """Cette fonction permet de créer les tables qui simule le retrait d'une liste de produits"""
        self.remove_elements("NOM_PRODUIT", produits)
        

    def remove_gammes(self, gammes : list[str]):
        """Cette fonction permet de créer les tables qui simule le retrait d'une liste de gammes"""
        self.remove_elements("GAMME", gammes)


    def remove_types(self, types : list[str]):
        """
        Cette fonction permet de créer les tables qui simule le retrait d'une liste de types.
        Pour le moment il n y a que deux type 'F&B' et 'Non F&B', donc la liste ne peut contenir qu'un seul élément sinon ça n'aura pas d'intérêt. 
        On garde la fonction dans ce format pour rester alligné avec les autres fonctions.sim.pipeline_file_path
        """
        self.remove_elements("TYPE", types)


    @classmethod
    def main(cls):
        """Cette fonction représente la fonction principale qui exploite cette classe et qu'elle faut exécuter"""

        # on s'assure qu'on a d'abord les données de modélisation de base avant de commencer la simulation
        super().main()

        sim = SalesPilSim() # instancier un objet de la classe de simulation

        
        sim.p_table_view("v_produits") # s'assurer que la vue sur les produits existe
        produits = sim.con.sql("SELECT DISTINCT produit from v_produits").df()["produit"] # récupérer les produits distincts indépendamment de l'hotel
        
        for produit in produits: # parcourir la liste des produits pour faire une simulation un par un 
            sim = SalesPilSim() # repartir d'une nouvelle instance avec les vues de base 
            sim.remove_produits([produit]) # simuler la suppression 
            break

In [11]:
SalesPilSim().main()

(130566, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(109342, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(872, 103)


,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,nombre_mois,produit_nombre_ventes_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,NON-F&B,SOUVENIRS,Tote-Bag Parisienne,6.0,192.0,192.0,0.0,32.0,32.00,0.00,24,0.250000,8.00000,8.00000,0.000,13.0,3,329.0,329.0,-2.131628e-14,25.307692,25.307692,-1.639714e-15,4.333333,109.666667,109.666667,-7.105427e-15,0.541667,13.708333,13.708333,-8.881784e-16,218.0,48,5,3435.4402,3495.10225,-59.66205,15.758900,16.032579,-0.273679,4.541667,71.571671,72.814630,-1.242959,43.60,687.08804,699.02045,-11.93241,9.083333,143.143342,145.629260,-2.485919,15759.0,146,9,2,84373.3057,83953.10225,420.20345,5.353976,5.327312,0.026664,107.938356,577.899354,575.021248,2.878106,1751.000000,9374.811744,9328.122472,46.689272,7879.5,42186.65285,41976.551125,210.101725,0.041096,24.333333,2626.500000,14062.217617,13992.183708,70.033908,0.006849,656.625000,3515.554404,3498.045927,17.508477,0.020548,0.328767,0.000381,0.002276,0.002287,0.000000,0.000825,0.003899,0.003919,-5.072848e-17,0.013833,0.040717,0.041632,-0.141984
1,H2075,Ibis budget Nice,simply,6.0,NON-F&B,SOS,Spray Répulsif Anti Moustique et Tique Icaridine - 60 Ml,5.0,40.0,40.0,0.0,8.0,8.00,0.00,29,0.172414,1.37931,1.37931,0.000,25.0,5,251.5,281.0,-2.950000e+01,10.060000,11.240000,-1.180000e+00,5.000000,50.300000,56.200000,-5.900000e+00,0.862069,8.672414,9.689655,-1.017241e+00,730.0,97,5,7319.5600,7360.68000,-41.12000,10.026795,10.083123,-0.056329,7.525773,75.459381,75.883299,-0.423918,146.00,1463.91200,1472.13600,-8.22400,25.172414,252.398621,253.816552

2026-08-04 05:33:14,213 | DEBUG | process(v_sales)
2026-08-04 05:33:14,220 | DEBUG | process(v_sales_model)
2026-08-04 05:33:14,224 | DEBUG | process(v_produits)
2026-08-04 05:33:14,333 | DEBUG | process(v_refs)
2026-08-04 05:33:14,345 | DEBUG | process(v_produit_ventes)
2026-08-04 05:33:14,354 | DEBUG | process(v_nombre_mois)
2026-08-04 05:33:14,360 | DEBUG | process(v_produit_ventes_mois)
2026-08-04 05:33:14,376 | DEBUG | process(v_gamme_ventes)
2026-08-04 05:33:14,385 | DEBUG | process(v_gamme_ventes_mois)
2026-08-04 05:33:14,399 | DEBUG | process(v_type_ventes)
2026-08-04 05:33:14,408 | DEBUG | process(v_type_ventes_mois)
2026-08-04 05:33:14,420 | DEBUG | process(v_ventes)
2026-08-04 05:33:14,430 | DEBUG | process(v_ventes_mois)
2026-08-04 05:33:14,438 | DEBUG | process(v_sales_model_base)


In [13]:
sim = SalesPilSim() # instancier un objet de la classe de simulation
sim.p_table_view("v_sales_model_base").df().head(3)

,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,nombre_mois,produit_nombre_ventes_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,F&B,SUGARY FOOD,Set de Thé (3 x 25g),4.073863,48.886351,48.886351,0.0,12.000000,12.000000,0.000000,24,0.169744,2.036931,2.036931,0.000000,2271.178374,24,5634.324543,5646.373501,-12.048958,2.480793,2.486099,-0.005305,94.632432,234.763523,235.265563,-0.502040,94.632432,234.763523,235.265563,-0.502040,15531.60099,97,5,80404.623962,79869.093851,535.530111,5.176841,5.142361,0.034480,160.119598,828.913649,823.392720,5.520929,3106.320198,16080.924792,15973.818770,107.106022,647.150041,3350.192665,3327.878910,22.313755,15753.626499,145,9,2,83903.50176,83428.735396,474.766364,5.325980,5.295843,0.030137,108.645700,578.644840,575.370589,3.274251,1750.402944,9322.611307,9269.859488,52.751818,7876.81325,41951.75088,41714.367698,237.383182,0.041379,24.166667,2625.604417,13983.916960,13904.789233,79.127727,0.006897,656.401104,3495.979240,3476.197308,19.781932,0.165517,0.668966,0.000259,0.000583,0.000586,0.000000,0.144169,0.067152,0.067679,-0.025379,0.985906,0.958299,0.957333,1.127987
1,H2075,Ibis budget Nice,simply,6.0,F&B,SUGARY FOOD,Tagada Haribo 120g,6.000000,21.000000,18.000000,3.0,3.500000,3.000000,0.500000,29,0.206897,0.724138,0.620690,0.103448,165.000000,8,298.200000,440.500000,-142.300000,1.807273,2.669697,-0.862424,20.625000,37.275000,55.062500,-17.787500,5.689655,10.282759,15.189655,-4.906897,4330.00000,38,3,10318.520000,14911.400000,-4592.880000,2.383030,3.443741,-1.060711,113.947368,271.540000

In [14]:
produits = ["Tongs Femme 100 Noir"]
champs = "NOM_PRODUIT"
list_str = ",".join([f"'{produit}'" for produit in produits])
sim.remove_produits(produits)

2026-08-04 05:33:52,036 | DEBUG | process(v_refs)
2026-08-04 05:33:52,050 | DEBUG | process(v_produit_ventes)
2026-08-04 05:33:52,070 | DEBUG | process(v_nombre_mois)
2026-08-04 05:33:52,076 | DEBUG | process(v_produit_ventes_mois)
2026-08-04 05:33:52,094 | DEBUG | process(v_gamme_ventes)
2026-08-04 05:33:52,104 | DEBUG | process(v_gamme_ventes_mois)
2026-08-04 05:33:52,123 | DEBUG | process(v_type_ventes)
2026-08-04 05:33:52,132 | DEBUG | process(v_type_ventes_mois)
2026-08-04 05:33:52,153 | DEBUG | process(v_ventes)
2026-08-04 05:33:52,163 | DEBUG | process(v_ventes_mois)
2026-08-04 05:33:52,173 | DEBUG | process(v_sales_model_base)


In [13]:
sim.close_con()